## Email Subject Line Generator (DistilGPT-2)

**Author:** Visshva R  
Core logic lives in `src/email_subject_generator/`. This notebook is a thin demo.

1. Load DistilGPT-2 via the package
2. Pull a small AESLC sample
3. Generate subjects with best-of-N scoring
4. Report ROUGE against human subject lines

In [ ]:
import os
import sys
from pathlib import Path

os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

from IPython.display import display
import pandas as pd

from email_subject_generator.config import MODEL_NAME, OUTPUT_DIR, PROJECT_ROOT
from email_subject_generator.data import load_aeslc_sample
from email_subject_generator.evaluation import compute_rouge, summarize_metrics
from email_subject_generator.generator import SubjectGenerator
from email_subject_generator.preprocessing import naive_subject

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Outputs     :", OUTPUT_DIR)

In [ ]:
print("Loading model (first run downloads DistilGPT-2)...")
gen = SubjectGenerator()
print("Loaded:", MODEL_NAME)

In [ ]:
df = load_aeslc_sample(n=8, seed=42)
print(f"Loaded {len(df)} AESLC rows")
display(df.head())

In [ ]:
smoke = "Email about an upcoming product launch with a keynote speaker"
print("Smoke test:", gen.generate_subject(smoke))
print("Best-of-N :", gen.best_of_n(smoke, n=3, k=2, min_overlap=0))

In [ ]:
from tqdm import tqdm

preds, naive = [], []
for desc in tqdm(df["description"], total=len(df)):
    preds.append(gen.best_of_n(desc, n=3, k=1, min_overlap=0)[0])
    naive.append(naive_subject(desc))

df = df.copy()
df["generated_subject"] = preds
df["naive_subject"] = naive

rouge = compute_rouge(preds, df["actual_subject"].tolist())
naive_rouge = compute_rouge(naive, df["actual_subject"].tolist())
print("Best-of-N ROUGE-1/2/L:", {k: round(v, 4) for k, v in rouge.items()})
print("Naive     ROUGE-1/2/L:", {k: round(v, 4) for k, v in naive_rouge.items()})
print(summarize_metrics(preds, rouge))
display(df[["description", "actual_subject", "generated_subject", "naive_subject"]].head())